# Module 1: Dimensional Model for DDC Epidemiological Data Warehouse

## Learning Objectives
By the end of this module, you will be able to:
1. Understand the star schema design for DDC's surveillance system
2. Explore dimension and fact tables using SQL
3. Write star join queries to analyze infection data
4. Understand how SCD Type 2 handles changing administrative boundaries

## Setup

Import the helper function to run SQL queries against Vertica:

In [ ]:
from ddc_helpers import run_query

## Problem Statement

The Department of Disease Control (DDC) receives **over 7,000 daily infection reports** from subdistricts across Thailand. Every few years, the government redraws administrative boundaries. Last year alone, **47 subdistricts were merged or split**.

**Challenge:** How do we track dengue trends across boundaries that keep changing?

**Solution:** A dimensional model with **SCD Type 2** on the location dimension.

A star schema lets us:
1. Slice infection data by time, geography, and disease independently
2. Handle boundary changes WITHOUT breaking historical queries
3. Leverage Vertica's columnar storage for fast aggregation

### The Star Schema

```
      dim_date ──┐
    dim_disease ──┼── fact_infection
   dim_location ──┘
```

---
## Dimension 1: dim_date

**Why a date dimension?** Because `GROUP BY MONTH(infection_date)` is slow. A pre-built date dimension lets Vertica use projection ordering for fast time-series aggregation. We also store Thai holidays for DDC analysis.

The date dimension contains **731 rows** covering 2023-2024 (365 + 366 days).

### Explore the first 10 dates

In [ ]:
run_query("""
SELECT *
FROM ddc_training.dim_date
ORDER BY date_id
LIMIT 10
""")

### Verify total date range

In [ ]:
run_query("""
SELECT
    COUNT(*) AS total_days,
    MIN(full_date) AS first_date,
    MAX(full_date) AS last_date
FROM ddc_training.dim_date
""")

### List Thai holidays

In [ ]:
run_query("""
SELECT
    full_date,
    day_name_en,
    thai_holiday_name
FROM ddc_training.dim_date
WHERE thai_holiday_name IS NOT NULL
ORDER BY full_date
""")

---
## Dimension 2: dim_disease

Five diseases that DDC tracks operationally. The `pathogen_type` and `transmission_mode` columns enable outbreak-pattern analysis.

In [ ]:
run_query("""
SELECT *
FROM ddc_training.dim_disease
ORDER BY disease_sk
""")

---
## Dimension 3: dim_location (SCD Type 2)

This is the heart of our spatial data warehouse.

### Key Design Decisions:
- **location_sk**: Surrogate key (IDENTITY). NEVER use `subdistrict_code` as PK because the same code can appear twice (before/after boundary change).
- **start_date / end_date / is_current**: SCD Type 2 tracking columns.
  - `end_date = '9999-12-31'` means "current version."
- **subdistrict_geom**: Vertica GEOMETRY column storing the polygon boundary.

We load **5 real Thai subdistricts** across different provinces.

In [ ]:
run_query("""
SELECT
    location_sk,
    subdistrict_code,
    subdistrict_name_th,
    province_name_th,
    start_date,
    end_date,
    is_current
FROM ddc_training.dim_location
ORDER BY location_sk
""")

---
## Fact Table: fact_infection

**Grain:** One row per patient per infection event.

### Design Notes:
- `patient_id`: Natural key from DDC's R506 surveillance system
- `infection_date_id` and `diagnosis_date_id`: FK to `dim_date` (YYYYMMDD INT format)
- `location_sk`: FK to `dim_location` (the SCD-aware surrogate key)
- `disease_sk`: FK to `dim_disease`
- `severity_score`: 1-5 scale (5 = fatal)
- `outcome`: 'recovered', 'hospitalized', 'deceased'

### Count total infections

In [ ]:
run_query("""
SELECT COUNT(*) AS total_infections
FROM ddc_training.fact_infection
""")

### Preview fact table records

In [ ]:
run_query("""
SELECT *
FROM ddc_training.fact_infection
LIMIT 10
""")

---
## The Star Join

This is **the fundamental query pattern** for DDC dashboards. We join all four tables to get a complete view of infection events with human-readable dimensions.

In [ ]:
run_query("""
SELECT
    dd.full_date,
    dd.month_name_th,
    dis.disease_name_th,
    loc.subdistrict_name_th,
    loc.province_name_th,
    fi.patient_id,
    fi.severity_score,
    fi.outcome
FROM ddc_training.fact_infection fi
JOIN ddc_training.dim_date dd ON fi.infection_date_id = dd.date_id
JOIN ddc_training.dim_disease dis ON fi.disease_sk = dis.disease_sk
JOIN ddc_training.dim_location loc ON fi.location_sk = loc.location_sk
WHERE fi.patient_id LIKE 'PID-2023-%'
ORDER BY dd.full_date
""")

---
## Aggregation: Monthly Cases by Disease

This query demonstrates how to aggregate infection data by time and disease, calculating case counts and average severity.

In [ ]:
run_query("""
SELECT
    dd.year_num,
    dd.month_num,
    dd.month_name_th,
    dis.disease_name_en,
    COUNT(*) AS case_count,
    ROUND(AVG(fi.severity_score), 1) AS avg_severity
FROM ddc_training.fact_infection fi
JOIN ddc_training.dim_date dd ON fi.infection_date_id = dd.date_id
JOIN ddc_training.dim_disease dis ON fi.disease_sk = dis.disease_sk
GROUP BY dd.year_num, dd.month_num, dd.month_name_th, dis.disease_name_en
ORDER BY dd.year_num, dd.month_num, dis.disease_name_en
""", limit=30)

---
## Key Concepts Summary

| Component | Purpose | Key Features |
|-----------|---------|-------------|
| **dim_date** | Time dimension | 731 rows (2023-2024), Thai holidays, pre-computed date attributes |
| **dim_disease** | Disease catalog | 5 diseases with pathogen type and transmission mode |
| **dim_location** | Geographic dimension | SCD Type 2 for boundary changes, surrogate key (location_sk) |
| **fact_infection** | Infection events | Grain: one patient per infection, FKs to all dimensions |
| **Star Join** | Core query pattern | JOIN fact to all dimensions for complete analysis |

---
## Exercise: Top 5 Provinces by Infections

**Task:** Find the top 5 provinces by total infections in 2023, showing disease breakdown.

**Requirements:**
1. Filter for year 2023 only
2. Group by province and disease
3. Calculate total cases per province-disease combination
4. Order by total cases descending
5. Limit to 30 rows

Fill in the blanks below:

In [ ]:
# Your solution here
run_query("""
SELECT
    loc.province_name_th,
    dis.disease_name_th,
    COUNT(*) AS total_cases
FROM ddc_training.fact_infection fi
JOIN ddc_training.dim_date dd ON fi.infection_date_id = dd.date_id
JOIN ddc_training.dim_disease dis ON fi.disease_sk = ___________
JOIN ddc_training.dim_location loc ON fi.location_sk = ___________
WHERE dd.year_num = ___________
GROUP BY ___________, ___________
ORDER BY total_cases ___________
""", limit=30)

<details>
<summary><b>Click to reveal answer</b></summary>

```sql
SELECT
    loc.province_name_th,
    dis.disease_name_th,
    COUNT(*) AS total_cases
FROM ddc_training.fact_infection fi
JOIN ddc_training.dim_date dd ON fi.infection_date_id = dd.date_id
JOIN ddc_training.dim_disease dis ON fi.disease_sk = dis.disease_sk
JOIN ddc_training.dim_location loc ON fi.location_sk = loc.location_sk
WHERE dd.year_num = 2023
GROUP BY loc.province_name_th, dis.disease_name_th
ORDER BY total_cases DESC
```

</details>

---
## Next Steps

In **Module 2**, we'll explore how Vertica projections make these star join queries **50x faster** through columnar storage and smart encoding.